# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ayazahmedbscsf23-art/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

I want to investigate which pages should be reviewed first because their organic-search performance may be declining or their content may be getting stale. This lane connects the available page-level search and engagement signals to a practical decision: which pages deserve a human content review first. It is a useful seven-week project because the starter data includes impressions, clicks/CTR, average position, trend, content age, and engagement signals. I will compare a transparent rule-based shortlist with a simple ML ranking, and keep the final output as recommendations for human review—not automatic content changes. I can change lanes by Week 4 if the data checks show a better question

In [11]:
# Load the anonymized starter CSV and show reproducible evidence.
# In Colab, clone the starter repo only if the data file is not already present.
from pathlib import Path
import os
import subprocess
import sys
import pandas as pd

DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")

if not DATA_PATH.exists():
    repo_dir = Path("flyrank-ml-internship-starter")
    if not (repo_dir / "data/raw/content_refresh_anonymized.csv").exists():
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/flyrank-bih/flyrank-ml-internship-starter.git",
             str(repo_dir)],
            check=True
        )
    DATA_PATH = repo_dir / "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)
print(f"Starter dataset size: {df.shape[0]:,} rows × {df.shape[1]} columns")

# Two useful data-based numbers: share of pages with a down trend and median 90-day impressions.
if "trend_direction" in df.columns:
    down_share = df["trend_direction"].astype(str).str.lower().eq("down").mean() * 100
    print(f"Pages marked with a down trend: {down_share:.1f}%")
if "impressions_90d" in df.columns:
    median_impressions = df["impressions_90d"].median()
    print(f"Median impressions over 90 days: {median_impressions:,.1f}")

# Optional third number: pages with meaningful visibility and a down trend.
if {"trend_direction", "impressions_90d"}.issubset(df.columns):
    visible_down = (
        df["trend_direction"].astype(str).str.lower().eq("down")
        & (pd.to_numeric(df["impressions_90d"], errors="coerce") >= 100)
    ).sum()
    print(f"Visible pages with a down trend (impressions_90d >= 100): {visible_down:,}")


Starter dataset size: 30,000 rows × 44 columns
Pages marked with a down trend: 54.2%
Median impressions over 90 days: 731.0
Visible pages with a down trend (impressions_90d >= 100): 13,152


## 2. The question: decision, action, cost of a wrong call

Search question: Among anonymized pages with enough organic-search visibility, which pages should a content team review first because they show signals of declining performance or a possible content-refresh opportunity?
- Decision improved: how to prioritize a limited number of pages for review this week.
- Unit of analysis: one anonymized page (one row in the starter CSV).
- Output: a ranked review queue with a score and plain-language reason codes, such as declining trend, high impressions, stale content, or low CTR relative to its position.
- Who acts and what they do: an SEO/content specialist checks the highest-ranked pages, then decides whether to update the title/meta description, improve intent match or page content, investigate engagement, or simply monitor the page.
- Cost of a wrong call: a false positive wastes review and editing time and may lead to an unnecessary change; a false negative means a genuinely useful refresh opportunity is missed. A poorly chosen change could also hurt performance, so the system should recommend review rather than automatically edit or publish.
- Why data/ML may help: there are multiple signals and many pages, so a consistent ranking may help humans prioritize. I will test whether a model improves on a transparent rule-based baseline using a held-out evaluation, rather than assuming ML is automatically better.

In [12]:
# Confirm the unit of analysis and key fields available in the actual CSV.
print("Unit of analysis: one row/page")
print("Columns relevant to the question:")
candidate_cols = [
    "impressions_90d", "clicks_90d", "ctr", "avg_position",
    "trend_direction", "content_age_days", "days_since_last_update",
    "sessions_90d", "engagement_rate", "scroll_rate"
]
print([c for c in candidate_cols if c in df.columns])


Unit of analysis: one row/page
Columns relevant to the question:
['impressions_90d', 'clicks_90d', 'ctr', 'avg_position', 'trend_direction', 'content_age_days', 'days_since_last_update', 'sessions_90d', 'engagement_rate', 'scroll_rate']


## 3. Quick look at the data (2-3 real numbers)

The code cells load the anonymized starter CSV directly and calculate the evidence below from the rows available in this checkout. I will use the printed row/column count, down-trend share, and visibility count to judge whether the lane has enough observable cases to investigate. These are descriptive numbers, not proof that a content update will cause better rankings or traffic.

For context, the repository's documented starter evaluation reports Precision@50 = 0.240 for baseline rules and 0.740 for the random forest, with random-forest ROC AUC = 0.750. These are reported starter-pipeline results, not results from my own experiment; I will verify the pipeline and validation setup before relying on them.

In [13]:
# A compact, data-driven summary plus a reproducible check of documented starter metrics.
print("Framing evidence from the loaded starter CSV:")
print(f"- Rows/pages: {len(df):,}")
if "trend_direction" in df.columns:
    trend = df["trend_direction"].astype(str).str.lower()
    print(f"- Down-trend pages: {trend.eq('down').sum():,} ({trend.eq('down').mean()*100:.1f}%)")
if {"trend_direction", "impressions_90d"}.issubset(df.columns):
    impressions = pd.to_numeric(df["impressions_90d"], errors="coerce")
    eligible = trend.eq("down") & impressions.ge(100)
    print(f"- Down-trend pages with at least 100 impressions: {eligible.sum():,}")

# Published reference numbers from the repository's starter evaluation (not recalculated here).
starter_reference = {
    "baseline_rules_precision_at_50": 0.240,
    "random_forest_precision_at_50": 0.740,
    "random_forest_roc_auc": 0.750,
}
print("\nDocumented starter-pipeline reference metrics (for context, not my own run):")
for metric, value in starter_reference.items():
    print(f"- {metric}: {value:.3f}")


Framing evidence from the loaded starter CSV:
- Rows/pages: 30,000
- Down-trend pages: 16,262 (54.2%)
- Down-trend pages with at least 100 impressions: 13,152

Documented starter-pipeline reference metrics (for context, not my own run):
- baseline_rules_precision_at_50: 0.240
- random_forest_precision_at_50: 0.740
- random_forest_roc_auc: 0.750


## 4. Careful words: what I can and can't claim

What I may claim after testing: I can report observed patterns in this anonymized starter sample and measure how well a rule or model ranks the chosen label under the stated validation setup. If a model beats the baseline on held-out clients, I can say it showed better ranking performance on that evaluation.

What I cannot claim: I cannot say the model proves why a page declined, that refreshing a page will cause more clicks, that it predicts Google's ranking algorithm, or that the results generalize to every website or the full warehouse. Search position, seasonality, competition, page intent, and other factors may affect outcomes. Low CTR alone does not prove that a title is bad, and low-volume pages can be noisy.

Safeguards: use client-holdout validation where available; avoid features that leak the label or use future information; compare pages within similar position tiers when studying CTR; show reason codes and evidence volume; and leave the final action to a human reviewer. Treat the score as decision support, not a guarantee.

In [14]:
# Self-check helpers: these checks do not replace my written judgment.
assert len(df) > 0, "The starter CSV loaded but contains no rows."
assert "impressions_90d" in df.columns, "Expected impressions_90d field is missing."
print("PASS: starter CSV loaded and the core visibility field is present.")
print("Reminder: manually verify the notebook has no client names, URLs, or private queries.")
print("Reminder: run all cells top-to-bottom in Colab, review the printed numbers, then save/commit to work/notebooks/w01_research_question.ipynb.")


PASS: starter CSV loaded and the core visibility field is present.
Reminder: manually verify the notebook has no client names, URLs, or private queries.
Reminder: run all cells top-to-bottom in Colab, review the printed numbers, then save/commit to work/notebooks/w01_research_question.ipynb.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.